In [ ]:
import scanpy as sc
import omicverse as ov
import pandas as pd
import numpy as np
import os
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
# Paths relative to SpatialRNA; algorithm settings and source cells are unchanged.
for folder in ['prepared/spaceflow_tuning', 'prepared/r_transfer', 'results/Extended_Data_Figure_5/plots']:
    os.makedirs(folder, exist_ok=True)
sc.settings.figdir = 'results/Extended_Data_Figure_5/plots'

ov.utils.download_pathway_database()
pathway_dict=ov.utils.geneset_prepare('genesets/GO_Biological_Process_2021.txt',organism='Human')

In [ ]:
adata = sc.read_h5ad('data/bin50_polygon_select.h5ad')

In [ ]:
sc.pl.violin(adata,['n_genes_by_counts','total_counts','pct_counts_mt'])

In [ ]:
adata = adata[adata.obs['total_counts']>500]
adata = adata[adata.obs['n_genes_by_counts']>200]
adata = adata[adata.obs['pct_counts_mt']<5]

In [ ]:
sc.pl.embedding(adata,basis='spatial',color=['total_counts'])

In [ ]:
sc.pl.violin(adata,['n_genes_by_counts','total_counts'])

In [ ]:
for i in [2000,2500,3000]:
    for j in [45,50,60]:
        adata1 = ov.space.svg(adata,mode='prost',n_svgs=i,target_sum=1e4,platform="stereo-seq",)
        adata1 = adata1[:, adata1.var.space_variable_features]
        sf_obj = ov.space.pySpaceFlow(adata1)
        sf_obj.train(spatial_regularization_strength=0.1, 
                     z_dim=50, lr=1e-3, epochs=1000, 
                     max_patience=50, min_stop=100, 
                     random_seed=114514, gpu=0, 
                     regularization_acceleration=True, edge_subset_sz=1000000)
        sf_obj.cal_pSM(n_neighbors=j,resolution=1,
                        max_cell_for_subsampling=25000,psm_key='pSM_spaceflow')
        sc.pp.neighbors(adata1, n_pcs=50,
                       use_rep='spaceflow')
        ov.utils.cluster(adata1,use_rep='spaceflow',method='leiden',resolution=0.2)
        sc.pl.embedding(adata1,basis='spatial', color=['pSM_spaceflow','leiden'],save='svg'+str(i)+'_neighbors'+str(j)+'.pdf')
        adata1.write_h5ad('prepared/spaceflow_tuning/'+'svg'+str(i)+'_neighbors'+str(j)+'.h5ad')

In [ ]:
adata1 = sc.read_h5ad('prepared/spaceflow_tuning/svg3000_neighbors60.h5ad')

In [ ]:
sc.tl.leiden(adata1,resolution=0.22)
sc.pl.embedding(adata1,color=['leiden'],basis='spatial')

In [ ]:
adata2 = adata1[adata1.obsm['spatial'][:,1]>10000]
print(adata2.obs.leiden.value_counts())
sc.pl.embedding(adata2,color=['leiden'],basis='spatial')

In [ ]:
adata3 = adata1[adata1.obsm['spatial'][:,1]<10000]
print(adata3.obs.leiden.value_counts())
sc.pl.embedding(adata3,color=['leiden'],basis='spatial')

In [ ]:
sc.tl.umap(adata1)

In [ ]:
adata1.obs['sample'] = 'normal'
adata1.obs['sample'] = adata1.obs['sample'].tolist()
adata1.obs.loc[adata1.obsm['spatial'][:,1]<10000, 'sample'] = 'pregnant'

In [ ]:
adata1.obs['region'] = adata1.obs['leiden'].tolist()
adata1.obs.loc[adata1.obs['leiden'] == '0', 'region'] = 'White Pulp'
adata1.obs.loc[adata1.obs['leiden'] == '1', 'region'] = 'Red Pulp2'
adata1.obs.loc[adata1.obs['leiden'] == '3', 'region'] = 'Red Pulp2'
adata1.obs.loc[adata1.obs['leiden'] == '4', 'region'] = 'Red Pulp1'
adata1.obs.loc[adata1.obs['leiden'] == '5', 'region'] = 'Red Pulp1'
adata1.obs.loc[adata1.obs['leiden'] == '6', 'region'] = 'Red Pulp1'
adata1.obs.loc[adata1.obs['leiden'] == '2', 'region'] = 'Marginal Zone'

In [ ]:
sc.set_figure_params(figsize=[5,5],fontsize=10,dpi=100)

In [ ]:
sc.pl.umap(adata1,color=['sample','region'])

In [ ]:
sc.pl.embedding(adata1,color=['sample','region'],basis='spatial')

In [ ]:
### preprocess
adata.obs = adata1.obs
adata.obsm = adata1.obsm
adata.write_h5ad('prepared/processed_spleen.h5ad')

In [ ]:
adata

In [ ]:
from scipy.io import mmwrite
import pandas as pd
def write_10x(adata,outdir,out_meta = True,out_embed = True):
    mmwrite(outdir+"matrix.mtx", adata.X.T,field='integer')
    cell = pd.DataFrame(list(adata.obs_names))
    gene = pd.DataFrame(list(adata.var_names))
    cell.to_csv(outdir+'barcodes.tsv',index=False,header=False)
    gene.to_csv(outdir + 'features.tsv', index=False, header=False)
    if out_meta:
        meta = pd.DataFrame(adata.obs.to_dict())
        meta.to_csv(outdir + 'meta.csv')
    if out_embed:
        embed = pd.DataFrame(adata.obsm['X_umap'])
        embed.to_csv(outdir+'embed.csv')
    if out_embed:
        embed = pd.DataFrame(adata.obsm['spaceflow'])
        embed.to_csv(outdir+'scvi.csv')
write_10x(adata,'prepared/r_transfer/')